# ML Masterclass 04: Distance & Kernel Methods

Welcome to Notebook 04. In this module, we move away from calculating gradients of a loss function and instead look at **Geometry and Distance**.

We will explore two fundamental algorithms: K-Nearest Neighbors (KNN), which is pure distance, and Support Vector Machines (SVM), which uses geometry to draw the widest possible street between classes.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin. The answer involves Mercer's Theorem:

> *"The RBF (Gaussian) kernel mathematically projects your features into an infinite-dimensional space to find a linear boundary. How is it computationally possible for a standard laptop with 16GB of RAM to process an infinite-dimensional vector?"*

---

## Table of Contents
1. **The Curse of Dimensionality:** Why KNN fails in high dimensions.
2. **Support Vector Machines (SVM):** Maximum margins and the Hard/Soft Margin equations.
3. **The Kernel Trick:** Solving non-linear problems without actually adding dimensions.


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors use KNN for everything. Seniors know KNN is O(n) at inference time — for every new prediction, you must scan ALL training points. In production with 10M data points, each prediction takes seconds. Use approximate nearest neighbor (HNSW/Faiss) or switch to SVM which compresses the decision boundary into a few support vectors.

**Mental Model:** The Kernel Trick is like a dimension teleporter. Instead of actually mapping data into million-dimensional space (impossible), the kernel function computes what the dot product WOULD BE in that space, using only the original low-dimensional vectors. You get the benefit of high-dimensional separation at the cost of a simple function evaluation.

**Common Junior Pitfall:** Not scaling features before KNN or SVM. If Feature A is income [0, 1M] and Feature B is age [0, 100], the Euclidean distance is completely dominated by income. Age becomes invisible to the algorithm. Always StandardScale before distance-based methods.

---


## 1. The Curse of Dimensionality

K-Nearest Neighbors is beautiful because it makes no assumptions about the data distribution. It simply says: "You are what your neighbors are."

However, KNN has a fatal flaw in production when dealing with text, images, or high-feature datasets: **The Curse of Dimensionality**.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *As you increase the dimensionality of your dataset to millions of features, the concept of a 'nearest neighbor' breaks down. Why?*

**A:** In a very high-dimensional hypercube, almost all the volume is concentrated in the corners. The mathematical distance between *any* two random points approaches the exact same value. If every point in your dataset is exactly 10.001 units away from every other point, the concept of 'nearest' ceases to exist. Everything is equally far away.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

# -----------------------------------------------------
# IMPLEMENTATION: Proving the Curse of Dimensionality
# -----------------------------------------------------

def average_distance_in_dimensions(dims, n_points=500):
    """Calculates average and variance of distances between random points in D dimensions."""
    # Generate random points between 0 and 1
    points = np.random.rand(n_points, dims)
    
    # Calculate pairwise distances (inefficient loop for clarity)
    distances = []
    for i in range(n_points):
        for j in range(i+1, n_points):
            dist = np.linalg.norm(points[i] - points[j])
            distances.append(dist)
            
    return np.mean(distances), np.std(distances)

dimensions = [2, 10, 50, 100, 500, 1000]
means = []
stds = []

print("Calculating geometric distances in high dimensions...")
for d in dimensions:
    m, s = average_distance_in_dimensions(d)
    means.append(m)
    stds.append(s)
    print(f"{d}D Space -> Avg Distance: {m:.2f}, StdDev: {s:.4f}")

plt.figure(figsize=(8,3))
plt.plot(dimensions, stds, marker='o', color='red')
plt.title("The Curse of Dimensionality: Variance of distances collapses")
plt.xlabel("Number of Dimensions")
plt.ylabel("Standard Deviation of Distances")
plt.show()

# Notice how the Standard Deviation approaches 0 as dimensions increase.
# This means all distances become exactly the same!

## 2. SVM: The Maximum Margin Classifier

Regression tries to fit a line *through* the data. SVM tries to draw a line *between* the data, making the "street" as wide as possible.

The SVM Optimization objective (Primal form):
Minimize: $\frac{1}{2} ||w||^2$
Subject to: $y_i(w \cdot x_i + b) \ge 1$ for all $i$

In a **Soft Margin SVM**, we allow some points to be on the wrong side of the street to prevent overfitting. We add a penalty parameter $C$. 
*   High $C$: Strict. Narrow street. Very prone to overfitting.
*   Low $C$: Relaxed. Wide street. High bias, low variance.

In [ ]:
# -----------------------------------------------------
# IMPLEMENTATION: Linear SVM vs Logistic Regression Boundary
# -----------------------------------------------------
from sklearn.svm import SVC
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=50, centers=2, random_state=6, cluster_std=1.2)

# Fit a strict Linear SVM (High C)
clf = SVC(kernel='linear', C=1000)
clf.fit(X, y)

plt.figure(figsize=(8, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, s=30, cmap=plt.cm.Paired)

# Plot the decision function
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

xx = np.linspace(xlim[0], xlim[1], 30)
yy = np.linspace(ylim[0], ylim[1], 30)
YY, XX = np.meshgrid(yy, xx)
xy = np.vstack([XX.ravel(), YY.ravel()]).T
Z = clf.decision_function(xy).reshape(XX.shape)

# plot decision boundary and margins
ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1], alpha=0.5,
           linestyles=['--', '-', '--'])
# plot support vectors
ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1], s=100,
           linewidth=1, facecolors='none', edgecolors='k')
plt.title("SVM Maximum Margin ('The Street') and Support Vectors")
plt.show()

## 3. The Kernel Trick 

What if the data cannot be separated by a straight line? We must map the data into a higher dimension where a hyperplane *can* slice it.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *The RBF kernel projects data into infinite dimensions. How does a computer compute this without crashing?*

**A:** Because it **never actually calculates the projection**. 

When you convert the SVM objective function into its **Dual Formulation**, the data points $x_i$ and $x_j$ only ever appear as a **Dot Product** $(x_i \cdot x_j)$. 

Mercer's Theorem states that for certain transformations $\phi(x)$, the dot product of the transformed vectors in the high-dimensional space is mathematically identical to sticking the original, low-dimensional vectors into a simple function (a Kernel). 

$\phi(x_a) \cdot \phi(x_b) = K(x_a, x_b)$

The RBF Kernel is simply: $K(x_a, x_b) = \exp(-\gamma ||x_a - x_b||^2)$. 
You just plug two 2D vectors into this fast exponential equation, and the resulting scalar is EXACTLY what the dot product would have been if you had physically transformed them into an infinite-dimensional space first! This is the most beautiful computational "trick" in Machine Learning.

---
## ✅ Knowledge Check

### Q1: Curse of Dimensionality

<details><summary>👉 Answer</summary>

In high-dimensional space, ALL points become equidistant. The ratio of (max distance - min distance) / min distance converges to 0 as dimensions increase. When every point is equally far from every other point, the concept of 'nearest neighbor' is meaningless — you're essentially picking randomly. This is why KNN fails on raw text (10,000+ dimensions) without dimensionality reduction.
</details>

### Q2: SVM C parameter

<details><summary>👉 Answer</summary>

High C = narrow margin, strict boundary, low bias, high variance (overfitting). Low C = wide margin, relaxed boundary, high bias, low variance (underfitting). C controls the trade-off between maximizing the margin width and minimizing training classification errors. In production, tune C via cross-validation.
</details>

### Q3: Kernel choice

<details><summary>👉 Answer</summary>

Linear kernel: use when data is linearly separable or n_features >> n_samples (text classification). RBF kernel: use when data needs nonlinear boundaries and n_samples is moderate. Poly kernel: rarely used in practice — RBF is almost always better. Key insight: RBF is really computing similarity (Gaussian distance) between points, so it's essentially a learned nearest-neighbor approach.
</details>


---
## 🔨 Practical Practice

### Exercise 1: Prove the Curse of Dimensionality empirically: generate random points in 2D, 10D, 100D, 1000D space. Plot the ratio of (max_distance - min_distance) / min_distance as dimensions increase. Show it approaches zero.

### Exercise 2: Implement a simple linear SVM from scratch using gradient descent on the hinge loss: L = max(0, 1 - y*(w·x + b)) + λ||w||². Compare your decision boundary to sklearn's SVC.

### Exercise 3: Visualize the kernel trick: create a 2D dataset that is NOT linearly separable (concentric circles). Show that mapping to 3D using φ(x₁,x₂) = (x₁², x₂², √2·x₁·x₂) makes it linearly separable.
